# 05 — Tuning Threshold dari Validation

**Tujuan:** memilih ambang probabilitas yang menyeimbangkan precision dan recall kelas bully, hanya dari validation, lalu menyimpannya untuk evaluasi test.

**Prasyarat:** `models/indobert_real/val_predictions.csv` dan `models/indobert_aug/val_predictions.csv`.

**Estimasi waktu:** 1 menit.

## Glosarium

- **Threshold**: batas probabilitas; 0,5 bukan angka suci.
- **PR curve**: grafik trade-off precision/recall.
- **F1**: metrik pemilih threshold di sini.
- **Leakage**: memilih threshold dari test membuat laporan terlalu optimistis.

## 1. Muat probabilitas validation

Kita hanya membutuhkan label asli dan probabilitas kelas bully. Tidak ada
training ulang dan tidak ada test set. Notebook ini aman dijalankan berkali-kali.

In [ ]:
import json

import pandas as pd

from cyberbully.config import LABEL_COLUMN, MODELS_DIR, OUT_DIR

MODEL_DIRS = {
    "A": MODELS_DIR / "indobert_real",
    "B": MODELS_DIR / "indobert_aug",
}
frames = {}
for key, directory in MODEL_DIRS.items():
    path = directory / "val_predictions.csv"
    if path.exists():
        frames[key] = pd.read_csv(path)
print({key: len(frame) for key, frame in frames.items()})


## 2. Kurva precision-recall

Menurunkan threshold menaikkan recall (lebih banyak bully tertangkap) tetapi
biasanya menurunkan precision (lebih banyak false positive). Kurva ini membantu
melihat pilihan yang masuk akal untuk penggunaan, bukan sekadar mengejar F1.

In [ ]:
if frames:
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve

    colors = {"A": "#0e5f55", "B": "#a5233a"}
    fig, ax = plt.subplots(figsize=(6.4, 4))
    for key, frame in frames.items():
        precision, recall, _ = precision_recall_curve(frame[LABEL_COLUMN], frame["probability"])
        ax.plot(recall, precision, color=colors[key], label=f"Model {key}")
    ax.set_xlabel("recall kelas bully")
    ax.set_ylabel("precision kelas bully")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()
else:
    print("Dilewati: prediksi belum ada.")


## 3. Pilih threshold dan simpan

`choose_threshold` memindai 0,05–0,95 dan memilih F1 bully terbaik. Hasilnya
disimpan ke `out/model_a/threshold.json` dan `out/model_b/threshold.json` agar notebook evaluasi dan laporan memakai angka
yang sama. Ini keputusan validation, bukan test.

In [ ]:
from cyberbully.metrics import binary_metrics, choose_threshold

thresholds = {}
if frames:
    for key, frame in frames.items():
        result = choose_threshold(frame[LABEL_COLUMN], frame["probability"])
        chosen = float(result["threshold"])
        thresholds[key] = chosen
        out_dir = OUT_DIR / f"model_{key.lower()}"
        out_dir.mkdir(parents=True, exist_ok=True)
        comparison = {
            "threshold": chosen,
            "validation_metrics": result["metrics"],
            "baseline_threshold_0.5": binary_metrics(frame[LABEL_COLUMN], frame["probability"], threshold=0.5),
            "fallback": result.get("fallback"),
            "best_candidate": result.get("best_candidate"),
        }
        payload = json.dumps(comparison, indent=2) + "\n"
        (out_dir / "threshold.json").write_text(payload)
        model_dir = MODEL_DIRS[key]
        if model_dir.exists():
            (model_dir / "threshold.json").write_text(payload)
            print("  disimpan juga ke", model_dir / "threshold.json")
        print("Model", key, "threshold:", round(chosen, 2))
    print(json.dumps(thresholds, indent=2))
else:
    print("Dilewati: prediksi belum ada.")


## 4. Bandingkan dengan 0,5

Tabel ini memperlihatkan harga dari keputusan threshold: berapa FN/FP yang
ditukar. Untuk alat bantu moderasi, trade-off ini sebaiknya didiskusikan dengan
pemilik produk dan dicatat di laporan.

In [ ]:
if frames:
    rows = []
    for key, frame in frames.items():
        for label, value in (("0.5", 0.5), (f"tuned ({thresholds[key]:.2f})", thresholds[key])):
            metrics = binary_metrics(frame[LABEL_COLUMN], frame["probability"], threshold=value)
            rows.append({"model": f"Model {key}", "threshold": label, **{k: metrics[k] for k in ("precision_bully", "recall_bully", "f1_bully")}})
    print(pd.DataFrame(rows).round(3).to_string(index=False))
else:
    print("Dilewati: prediksi belum ada.")


## Ringkasan

- Threshold dipilih dari validation, bukan test.
- PR curve menunjukkan trade-off precision/recall.
- `out/model_a/threshold.json` dan `out/model_b/threshold.json` menjadi sumber angka untuk evaluasi test dan laporan.
- Test set tetap belum disentuh di notebook ini.